# Google MLCC: standalone Colab edition

Run the setup cell in a fresh Colab runtime, then run every cell from top to bottom. Setup downloads the original course data and installs compatible dependencies, including `google-ml-edu` (imported as `ml_edu`) for taxi/rice. No course ZIP is required. TensorFlow 2.21 and Keras 3.12+ adapt those exercises to current Python 3.13 Colab; the original TF 2.18/NumPy 2.0 pins do not support Python 3.13.

`USE_GOOGLE_DRIVE=True` saves measured results, models, and task execution evidence to `MyDrive/HaiLong_Colab_Results/01_google_mlcc`. Save this notebook in Drive and save it after running to retain cell outputs. The completion report requires every computational stage and full original data/epochs. Website quizzes and interactive exercises have separate progress.

Original source: https://raw.githubusercontent.com/google/eng-edu/main/ml/cc/exercises/fairness_income.ipynb


In [ ]:
# Standalone Google MLCC setup. Start a fresh Colab runtime, then Run all.
# No ZIP or eng-edu clone is required. Full original data/tasks only.
from pathlib import Path
import os, sys, subprocess, importlib.metadata as metadata
import hashlib, json, shutil, urllib.request, platform, uuid
from datetime import datetime, timezone
from packaging.requirements import Requirement

USE_GOOGLE_DRIVE = True  #@param {type:"boolean"}
QUICK_RUN = False  # retained for compatibility; full completion requires False
if QUICK_RUN:
    raise ValueError('Use QUICK_RUN=False to execute the full original exercise.')
NOTEBOOK_NAME = '05_fairness_income_colab.ipynb'
DEPENDENCY_PROFILE = 'fairness'
REQUIREMENTS = ['tensorflow==2.21.0', 'tf-keras==2.21.0', 'tensorflow-model-remediation==0.1.7.1', 'numpy>=1.26,<3', 'pandas>=2.2,<3', 'matplotlib>=3.9,<4', 'scikit-learn>=1.5,<2']
REQUIRED_TASKS = ['full_acs_data', 'baseline_training_10_epochs', 'baseline_test_predictions', 'task_1_group_metrics', 'task_2_positive_sensitive_groups', 'task_3_pack_complete_training_data', 'mindiff_training_10_epochs', 'mindiff_test_predictions', 'task_4_measured_comparison']
DATA_SOURCES = [{'filename': 'acsincome_raw_2018.csv', 'url': 'https://download.mlcc.google.com/mledu-datasets/acsincome_raw_2018.csv', 'sha256': '8b55d12b7986cd6c24b604f7681efbd09dcd23835d5c191fa56e6972d368d669', 'rows': 1664500}]
ORIGINAL_SOURCE = 'https://raw.githubusercontent.com/google/eng-edu/main/ml/cc/exercises/fairness_income.ipynb'
VISUALIZATION_ADAPTATION = 'scikit-learn metric tables and matplotlib charts replace historical TFMA/Fairness Indicators widgets'

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# TensorFlow 2.21 has Python 3.13 wheels; original TF 2.18 pins do not.
# Set Keras mode before importing TensorFlow or Keras.
if DEPENDENCY_PROFILE in ('google', 'fairness'):
    if DEPENDENCY_PROFILE == 'fairness':
        if any(n in sys.modules for n in ('tensorflow', 'keras', 'tf_keras')):
            if os.environ.get('TF_USE_LEGACY_KERAS') != '1':
                raise RuntimeError('Select Runtime > Disconnect and delete runtime, reconnect, then Run all: MinDiff needs legacy Keras selected before TensorFlow import.')
        os.environ['TF_USE_LEGACY_KERAS'] = '1'
    else:
        if os.environ.get('TF_USE_LEGACY_KERAS') == '1' and any(n in sys.modules for n in ('tensorflow', 'keras', 'tf_keras')):
            raise RuntimeError('Start a fresh runtime after fairness before running this Keras 3 notebook.')
        os.environ.pop('TF_USE_LEGACY_KERAS', None)
    os.environ['KERAS_BACKEND'] = 'tensorflow'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
os.environ['COURSE_SMOKE'] = '0'
os.environ['WANDB_MODE'] = 'offline'

# google-ml-edu is the distribution that provides the ml_edu import.
IMPORT_NAMES = {'google-ml-edu':'ml_edu', 'tf-keras':'tf_keras',
    'tensorflow-model-remediation':'tensorflow_model_remediation',
    'scikit-learn':'sklearn'}
def installed_version(package):
    try:
        return metadata.version(package)
    except metadata.PackageNotFoundError:
        return None
missing, loaded_changes = [], []
for text in REQUIREMENTS:
    req = Requirement(text)
    version = installed_version(req.name)
    if version is None or version not in req.specifier:
        missing.append(text)
        module = IMPORT_NAMES.get(req.name, req.name.replace('-', '_'))
        if module in sys.modules:
            loaded_changes.append(module)
if missing and IN_COLAB:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', *missing])
    if loaded_changes:
        raise RuntimeError('Installed required versions. Select Runtime > Restart session, then Run all again; old loaded modules: ' + ', '.join(loaded_changes))
elif missing:
    raise RuntimeError('Prepared for Colab. Missing compatible local packages: ' + ', '.join(missing))

if IN_COLAB:
    COURSE_ROOT = Path('/content/HaiLong_Colab_Exercises')
else:
    COURSE_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                        if (p / 'colab_runtime.py').is_file()), None)
    if COURSE_ROOT is None:
        raise RuntimeError('Run in Colab or inside HaiLong_Colab_Exercises.')
COURSE_DIR = COURSE_ROOT / '01_google_mlcc'
COURSE_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(COURSE_DIR)
DATA = COURSE_DIR / 'data'
DATA.mkdir(exist_ok=True)
DRIVE_DIR = None
if IN_COLAB and USE_GOOGLE_DRIVE:
    if not Path('/content/drive/MyDrive').is_dir():
        drive.mount('/content/drive', timeout_ms=300000)
    DRIVE_DIR = Path('/content/drive/MyDrive/HaiLong_Colab_Results/01_google_mlcc')
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
for folder in ('results', 'models'):
    target = COURSE_DIR / folder
    if DRIVE_DIR is not None:
        (DRIVE_DIR / folder).mkdir(exist_ok=True)
        if not target.exists():
            target.symlink_to(DRIVE_DIR / folder, target_is_directory=True)
    target.mkdir(exist_ok=True)
RESULTS, MODELS = COURSE_DIR / 'results', COURSE_DIR / 'models'
for spec in DATA_SOURCES:
    target = DATA / spec['filename']
    if not target.is_file():
        temporary = target.with_suffix(target.suffix + '.download')
        print('Downloading original Google course data:', spec['url'])
        urllib.request.urlretrieve(spec['url'], temporary)
        if hashlib.sha256(temporary.read_bytes()).hexdigest() != spec['sha256']:
            raise ValueError('Downloaded dataset differs from verified original: ' + spec['filename'])
        temporary.replace(target)
    if hashlib.sha256(target.read_bytes()).hexdigest() != spec['sha256']:
        raise ValueError('Existing data differs from the original course dataset: ' + str(target))

EXECUTED_TASKS = {}
COLAB_CONTEXT = {'notebook': NOTEBOOK_NAME, 'run_id': str(uuid.uuid4()),
    'started_at_utc': datetime.now(timezone.utc).isoformat(),
    'in_hosted_colab': IN_COLAB, 'quick_run': False,
    'python': platform.python_version(),
    'versions': {Requirement(r).name: installed_version(Requirement(r).name) for r in REQUIREMENTS},
    'original_source': ORIGINAL_SOURCE, 'data_sources': DATA_SOURCES,
    'visualization_adaptation': VISUALIZATION_ADAPTATION}
def mark_task(name, **evidence):
    if name not in REQUIRED_TASKS:
        raise KeyError(name)
    EXECUTED_TASKS[name] = {'recorded_at_utc': datetime.now(timezone.utc).isoformat(), **evidence}
    checkpoint = {**COLAB_CONTEXT, 'last_completed_stage': name,
        'updated_at_utc': datetime.now(timezone.utc).isoformat(),
        'missing_tasks': sorted(set(REQUIRED_TASKS) - set(EXECUTED_TASKS)),
        'task_evidence': EXECUTED_TASKS, 'course_website_completion_confirmed': False}
    (RESULTS / (Path(NOTEBOOK_NAME).stem + '_progress.json')).write_text(
        json.dumps(checkpoint, indent=2, allow_nan=False), encoding='utf-8')
    if DRIVE_DIR is not None:
        for folder in ('results', 'models'):
            source, target = COURSE_DIR / folder, DRIVE_DIR / folder
            if source.resolve() != target.resolve():
                shutil.copytree(source, target, dirs_exist_ok=True)
def finish_run(context):
    missing_tasks = sorted(set(REQUIRED_TASKS) - set(EXECUTED_TASKS))
    if missing_tasks:
        raise RuntimeError('Required exercise stages did not run: ' + ', '.join(missing_tasks))
    report = {**context, 'finished_at_utc': datetime.now(timezone.utc).isoformat(),
        'full_original_computational_tasks_executed': True,
        'course_website_completion_confirmed': False,
        'required_tasks': REQUIRED_TASKS, 'task_evidence': EXECUTED_TASKS,
        'note': 'Execution evidence covers this notebook. It does not certify website quizzes, interactive exercises, or personal reflection.'}
    path = RESULTS / (Path(NOTEBOOK_NAME).stem + '_completion.json')
    path.write_text(json.dumps(report, indent=2, allow_nan=False), encoding='utf-8')
    if DRIVE_DIR is not None:
        for folder in ('results', 'models'):
            source, target = COURSE_DIR / folder, DRIVE_DIR / folder
            if source.resolve() != target.resolve():
                shutil.copytree(source, target, dirs_exist_ok=True)
        print('Saved computational progress and artifacts to:', DRIVE_DIR)
    print('Full original computational stages executed:', NOTEBOOK_NAME)
    print('Completion evidence:', path)
    return report
print('Setup ready:', NOTEBOOK_NAME, '| full course data |', DEPENDENCY_PROFILE)

# Google MLCC fairness: full computational exercise

The original TensorFlow model, all 1,664,500 ACS rows, 10 baseline epochs, 10 MinDiff epochs, positive-label group subsets, and four exercise tasks are retained. Normalization uses training rows only.

This current Colab adaptation uses TensorFlow 2.21 with matching `tf-keras` and TensorFlow Model Remediation 0.1.7.1. It computes the original group AUC/FNR comparisons with scikit-learn tables and matplotlib charts. Historical TensorFlow Model Analysis/Fairness Indicators widgets are not installed and their UI workflow is not claimed complete. Numerical answers are derived from this run.

Use a fresh runtime and run setup before importing TensorFlow. The optional portable fairness notebook is an extra adaptation and does not replace this original computational exercise.


In [ ]:
#@title Copyright 2024 Google LLC. Double-click for license information.
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

This edition runs the computational tasks in a current Colab runtime, with legacy Keras selected before TensorFlow import.


# Colabs

Machine Learning Crash Course uses Colaboratory (Colab) notebooks for all programming exercises. Colab is Google's implementation of [Jupyter Notebook](https://jupyter.org/). For more information about Colabs and how to use them, go to [Welcome to Colaboratory](https://research.google.com/colaboratory).

# Addressing Bias and Fairness Issues in ML Models

This notebook uses an updated and reconstructed version of the [UCI Adult](https://archive.ics.uci.edu/dataset/2/adult) dataset called [ACSIncome](https://github.com/socialfoundations/folktables) to:

*   Train a model that predicts whether an individual's income is above $50,000 USD based on education, employment, and marital status, as well as *sensitive* attributes, such as age, sex, and race.
*   Compute commonly-identified fairness metrics to evaluate how the model performs across demographic groups.
*   Apply a model remediation technique to minimize the difference in error rates between demographic groups.









## Learning objectives

* Evaluate overall and demographic-group AUC, false-negative rate, and false-positive rate from actual test predictions.
* Use TensorFlow Model Remediation MinDiff to address group disparity and measure any tradeoff in this run.

The original lesson introduces TensorFlow Model Analysis and Fairness Indicators. This edition renders equivalent point-estimate tables and charts without the historical interactive widgets or their confidence intervals.


## Intended Use & Considerations

This notebook demonstrates how to mitigate undesirable biases in a trained model. In practice, the model and the prediction task in this notebook is not realistic. Rather, the emphasis here is the approach to evaluating the performance of a trained model and minimizing error rates between demographics groups.

With regard to the prediction task, there may circumstances where an individual's income may be used as a determining factor, such as obtaining a loan, acquiring insurance, applying for assistance programs, or possibly to advertise products. But in those cases, such businesses, institutions or organizations would not be looking to infer one's income using a machine learning model; they would instead obtain that information directly from the applicant, if possible, or rely on other signals to decide on the outcome.

With that said, **this notebook is for educational purposes only and it is not advisable to use this model in any context outside of this exercise**.

## Setup
The first setup cell installs TensorFlow, matching legacy Keras, TensorFlow Model Remediation, and metric/plotting libraries.


If setup requests a restart after installation, restart and run every cell again. Use a fresh runtime when switching between this legacy Keras notebook and taxi/rice.


In [ ]:
# The Colab setup installs matching TensorFlow and tf-keras, plus MinDiff.

Import TensorFlow Model Remediation MinDiff, then define group metrics and plots for the original fairness questions.


In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow_model_remediation import min_diff
from sklearn.metrics import roc_auc_score, confusion_matrix, accuracy_score
tf.keras.utils.set_random_seed(42)

def fairness_table(frame):
    rows=[]
    for name, group in [('All', frame), *list(frame.groupby('SEX'))]:
        y=group['PINCP'].to_numpy(); p=group['PRED'].to_numpy().reshape(-1)
        tn,fp,fn,tp=confusion_matrix(y,p>=0.5,labels=[0,1]).ravel()
        rows.append({'group':name,'count':len(y),'accuracy':accuracy_score(y,p>=0.5),
                     'auc':roc_auc_score(y,p) if len(np.unique(y))==2 else np.nan,
                     'false_negative_rate':fn/(fn+tp) if fn+tp else np.nan,
                     'false_positive_rate':fp/(fp+tn) if fp+tn else np.nan})
    return pd.DataFrame(rows)

import matplotlib.pyplot as plt

def plot_fairness_metrics(table, title):
    fig, axes = plt.subplots(1, 2, figsize=(11,4))
    for ax, metric, label in zip(axes, ['auc','false_negative_rate'], ['AUC','False-negative rate']):
        ax.bar(table['group'].astype(str), table[metric])
        ax.set_ylim(0,1)
        ax.set_ylabel(label)
        ax.set_title(title + ': ' + label)
    plt.tight_layout()
    plt.show()

def group_gap(table, metric):
    by_group = table.set_index('group')
    return float(by_group.loc['Female',metric] - by_group.loc['Male',metric])

## About the ACSIncome Dataset
  
ACSIncome is one of several datasets created by [Ding et al.](https://proceedings.neurips.cc/paper_files/paper/2021/file/32e54441e6382a7fbacbbbaf3c450059-Paper.pdf) as an alternative to [UCI Adult](https://archive.ics.uci.edu/dataset/2/adult). A few key details about ACSIncome:
*   The dataset contains 1,664,500 datapoints pulled from the 2018 United States–wide [American Community Survey](https://www.census.gov/programs-surveys/acs) (ACS) [Public Use Microdata Sample](https://www.census.gov/programs-surveys/acs/microdata.html) (PUMS) data sample.
*   All fifty US states and Puerto Rico are represented in this dataset.
*   Each row represents a person described by various features, including age, race, and sex, which correspond to protected categories in different domains under US anti-discrimination laws.
*   The dataset only includes individuals above 16 years old who worked at least 1 hour per week in the past year and had an income of at least $100 USD.

For more information on the dataset and how it was created to reconstruct UCI Adult, check out the following citations:

> Ding, Frances, Moritz Hardt, John Miller, and Ludwig Schmidt. "[Retiring adult: New datasets for fair machine learning.](https://proceedings.neurips.cc/paper_files/paper/2021/hash/32e54441e6382a7fbacbbbaf3c450059-Abstract.html)" Advances in neural information processing systems 34 (2021): 6478-6490.

> Sarah Flood, Miriam King, Renae Rodgers, Steven Ruggles, and J. Robert Warren (2020). Integrated Public Use Microdata Series, Current Population Survey: Version 8.0 [dataset]. Minneapolis, MN: IPUMS. https://doi.org/10.18128/D030.V8.0







In [ ]:
acs_df = pd.read_csv(DATA / 'acsincome_raw_2018.csv')
assert len(acs_df) == 1664500, 'Use all original ACS rows; a quick sample is not full completion.'
assert not acs_df.isna().any().any()
assert set(acs_df['SEX'].unique()) == {1.0,2.0}
mark_task('full_acs_data', rows=len(acs_df), columns=list(acs_df.columns))
display(acs_df.sample(5, random_state=42))

## Features

After importing the dataset, five random samples appear in a table in the output cell. Each sample represents an individual, with each column representing an aspect of the invidiual, such as their age, occupation, place of birth, and so forth.

The following table describes each feature column:

| Feature    | Description |
| -------- | ------- |
| AGEP | Age |
| COW | Class of worker (government employee, self-employed, private employee) |
| SCHL | Educational attainment (high school diploma, bachelor's degree, doctorate degree) |
| MAR  | Marital status |
| OCCP | Occuptation |
| POBP | Place of birth |
| RELP | Relationship to householder (husband or wife, housemate or roommate, nursing home, group home, etc.)  |
| WKHP | Usual hours worked per week in the past 12 months |
| SEX | Male or female |
| RAC1P | Recorded detailed race code |
| ST | US state code that represents the individual's location |
| PINCP | Total person's yearly income |

All of these features are represented numerically, though some of them correspond to a coded value. For example, for the `COW` (Class of worker) feature, `1.0` represents *an employee of a private for-profit company or business, or of an individual, for wages, salary, or commissions* and `2.0` represents *an employee of a private not-for-profit, tax-exempt, or charitable organization*. See [the supplemental section](https://proceedings.neurips.cc/paper_files/paper/2021/file/32e54441e6382a7fbacbbbaf3c450059-Supplemental.pdf) of [Ding et al.](https://proceedings.neurips.cc/paper_files/paper/2021/file/32e54441e6382a7fbacbbbaf3c450059-Paper.pdf) and the [ACS PUMS 2018 Data Dictionary](https://www2.census.gov/programs-surveys/acs/tech_docs/pums/data_dict/PUMS_Data_Dictionary_2018.pdf) for the full mapping of codes.

## Change Target Value to Binary
[As stated earlier](#scrollTo=TL5y5fY9Jy_x), the task is to predict whether the annual income of a US working adult is more than $50,000. The `PINCP` (total person's yearly income) column in the dataset represents the target variable; however, the value will need to be convereted into a binary. For each sample, an individual’s target label will be `1` if `PINCP` > `50000.0`, otherwise `0`.



In [ ]:
LABEL_KEY = 'PINCP'
LABEL_THRESHOLD = 50000.0

acs_df[LABEL_KEY] = acs_df[LABEL_KEY].apply(
    lambda income: 1 if income > LABEL_THRESHOLD else 0)

acs_df.sample(10)

## Defining Base Model
For the purposes of this exercise, a simple, lightly-tuned [`keras.Model`](https://www.tensorflow.org/api_docs/python/tf/keras/Model) (using the [Functional API](https://www.tensorflow.org/guide/keras/functional) for preprocessing) will be created and serve as the base model for this exercise. Note that while the focus for this exercise is the technique involved in addressing fairness concerns, the model architecture — as taught throughout MLCC — would be thoughfully chosen and hyperparameter tuning would be performed before attempting to address any fairness concerns that might arise in the model.

In [ ]:
inputs = {}
features = acs_df.copy()
features.pop(LABEL_KEY)

# Instantiate a Keras input node for each column in the dataset.
for name, column in features.items():
  if name != LABEL_KEY:
    inputs[name] = tf.keras.Input(
        shape=(1,), name=name, dtype=tf.float64)

# Stack the inputs as a dictionary and preprocess them.
def stack_dict(inputs, fun=tf.stack):
  values = []
  for key in sorted(inputs.keys()):
    values.append(tf.cast(inputs[key], tf.float64))

  return fun(values, axis=-1)

x = stack_dict(inputs, fun=tf.concat)

# Collect the features from the DataFrame, stack them together and normalize
# their values by passing them to the normalization layer.
normalizer = tf.keras.layers.Normalization(axis=-1)
normalizer_training_rows = features.sample(frac=0.8, random_state=200)
normalizer.adapt(stack_dict(dict(normalizer_training_rows)))

# Build the main body of the model using a normalization layer, two dense
# rectified-linear layers, and a single output node for classification.
x = normalizer(x)
x = tf.keras.layers.Dense(64, activation='relu')(x)
x = tf.keras.layers.Dense(32, activation='relu')(x)
outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)

# Put it all together using the Keras Functional API
base_model = tf.keras.Model(inputs, outputs)

## Configuring Base Model
Since this is a binary classification task, computing the cross-entropy loss between true labels and predicted labels will be sufficient for this exercise.

In [ ]:
# Define the metrics used to monitor model performance while training.
METRICS = [
  tf.keras.metrics.BinaryAccuracy(name='accuracy'),
  tf.keras.metrics.AUC(name='auc'),
]

# Configure the model for training using a stochastic gradient descent
# optimizer, cross-entropy loss between true labels and predicted labels, and
# the metrics defined above to evaluate the base model during training.
base_model.compile(
    optimizer='adam',
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=METRICS)

## Convert to tf.data.Dataset
Most of the exercises throughout MLCC use a [pandas DataFrame](https://developers.google.com/machine-learning/glossary/#pandas) directly as an input argument for [`Model.fit`](https://www.tensorflow.org/api_docs/python/tf/keras/Model#fit). But in this exercise, the dataset must be converted to [`tf.data.Dataset`](https://www.tensorflow.org/api_docs/python/tf/data/Dataset) because that's the requirement for [`MinDiffModel`](https://www.tensorflow.org/responsible_ai/model_remediation/api_docs/python/model_remediation/min_diff/keras/MinDiffModel), which will be introducted later. Fortunately, this conversion is simple to do, thanks to [`tf.convert_to_tensor`](https://www.tensorflow.org/api_docs/python/tf/convert_to_tensor).

The following helper function will be useful for preparing the dataset in subsequent code cells:

In [ ]:
# Helper function to convert a pandas DataFrame into a tf.Data.dataset object
# necessary for the purposes of this exercise.
def dataframe_to_dataset(dataframe):
  dataframe = dataframe.copy()
  labels = dataframe.pop(LABEL_KEY)
  dataset = tf.data.Dataset.from_tensor_slices(
      ((dict(dataframe), labels)))
  return dataset

## Finalize Training Set & Train Base Model
At this point, all that remains is splitting the dataset before training the base model.

**NOTE:** *The following cell may take approximately 10—15 minutes to run.*

In [ ]:
RANDOM_STATE = 200
BATCH_SIZE = 100
EPOCHS = 10

# Use the sample() method in pandas to split the dataset into a training set
# that represents 80% of the original dataset, then convert it to a
# tf.data.Dataset object, and finally train the model using the
# converted training set.
acs_train_df = acs_df.sample(frac=0.8, random_state=RANDOM_STATE)
acs_train_ds = dataframe_to_dataset(acs_train_df)
acs_train_batches = acs_train_ds.batch(BATCH_SIZE)

baseline_history = base_model.fit(acs_train_batches, epochs=EPOCHS, verbose=2)
assert len(baseline_history.epoch) == 10
pd.DataFrame(baseline_history.history).to_csv(RESULTS / 'tf_mindiff_baseline_history.csv', index=False)
base_model.save_weights(str(MODELS / 'tf_mindiff_baseline.weights.h5'))
mark_task('baseline_training_10_epochs', epochs=len(baseline_history.epoch), train_rows=len(acs_train_df))

## Evaluate Base Model
Consistent with [Ding et al.](https://proceedings.neurips.cc/paper_files/paper/2021/file/32e54441e6382a7fbacbbbaf3c450059-Paper.pdf), the overall accuracy for the base model should be at around 80% with minimal tuning and a basic model architecture. The following code cell uses the test set to evaluate the performance of the base model:

In [ ]:
# Use the indices from the training set to create the test set, which represents
# 20% of the original dataset; then convert it to a tf.data.Dataset object, and
# evaluate the base model using the converted test set.
acs_test_df = acs_df.drop(acs_train_df.index).sample(frac=1.0, random_state=RANDOM_STATE)
acs_test_ds = dataframe_to_dataset(acs_test_df)
acs_test_batches = acs_test_ds.batch(BATCH_SIZE)

assert len(acs_train_df) == 1331600 and len(acs_test_df) == 332900
assert set(acs_train_df.index).isdisjoint(acs_test_df.index)
assert set(normalizer_training_rows.index) == set(acs_train_df.index)
base_test_metrics = base_model.evaluate(acs_test_batches, return_dict=True, verbose=2)
pd.DataFrame([base_test_metrics]).to_csv(RESULTS / 'tf_mindiff_baseline_test_metrics.csv', index=False)

## Evaluating for Fairness
With the base model trained, now would be a good opportunity to evaluate performance across demographic groups. For ease of analysis, [Fairness Indicators](https://www.tensorflow.org/responsible_ai/fairness_indicators/guide) will be used to compute fairness metrics across demographic groups and visualize results.

To begin, a column containing all the base model's predictions from the test set will be needed in order to configure Fairness Indicators. [`model.predict(test_set)`](https://www.tensorflow.org/api_docs/python/tf/keras/Model#predict) will be used to generate the output predictions.

In [ ]:
# Generate output predictions using the test set.
base_model_predictions = base_model.predict(
    acs_test_batches, batch_size=BATCH_SIZE)

assert base_model_predictions.reshape(-1).shape == (len(acs_test_df),)
mark_task('baseline_test_predictions', test_rows=len(acs_test_df), test_metrics={k:float(v) for k,v in base_test_metrics.items()})

## A Note on Sensitive Attributes
There are several features in ACSIncome that can be used to evaluate for fairness. For this exercise, the `SEX` attribute was chosen. This is in part to keep the config terse.

The `SEX` attribute used by the [US Census Bureau surveys](https://www.census.gov/glossary/?term=Sex) to construct ACSIncome was specifically intended to capture an individual's biological sex and not gender. As such, possible ambiguity of these concepts could have tampered with their intended data collection, which could result in some misrepresentation. Ideally, there would be a separate category in such surveys that allow for an individual to express their gender identity (male, female, non-binary, agender, and so forth). But for the purposes of this exercise, the `SEX` attribute available in ACSIncome will suffice for demonstrating how to perform fairness evaluations.

In practice, the recommended approach is to evaluate across any group that may be negatively impacted by the trained model and is accessible in the dataset, which can include race, ethnicity, working status, educational background, and even the US state that the individual is in.




In [ ]:
SENSITIVE_ATTRIBUTE_VALUES = {1.0: 'Male', 2.0: 'Female'}
SENSITIVE_ATTRIBUTE_KEY = 'SEX'
PREDICTION_KEY = 'PRED'
base_model_analysis = acs_test_df.copy()
base_model_analysis[SENSITIVE_ATTRIBUTE_KEY] = base_model_analysis[SENSITIVE_ATTRIBUTE_KEY].map(SENSITIVE_ATTRIBUTE_VALUES)
base_model_analysis[PREDICTION_KEY] = base_model_predictions.reshape(-1)
display(base_model_analysis.sample(5, random_state=42))

## Compute group fairness metrics
Compute count, accuracy, AUC, FNR, and FPR for All, Male, and Female from the same held-out rows. Binary decisions use threshold 0.5. These are point estimates; no historical TFMA config or widget is needed in this adaptation.


In [ ]:
base_model_eval_result = fairness_table(base_model_analysis)

### Task 1: Identify fairness concerns
Inspect the computed table and AUC/FNR charts below.

1. How similar is AUC between demographic groups in this run?
2. How does the female FNR compare with the male rate and overall baseline?

Compare group counts as well as metrics. AUC can be similar while FNR differs at a fixed threshold.


In [ ]:
display(base_model_eval_result)
base_model_eval_result.to_csv('results/tf_mindiff_base_metrics.csv',index=False)

plot_fairness_metrics(base_model_eval_result, 'Baseline')
auc_gap_before = group_gap(base_model_eval_result, 'auc')
fnr_gap_before = group_gap(base_model_eval_result, 'false_negative_rate')
print(f'Task 1 measured answer: female minus male AUC = {auc_gap_before:+.4f}; female minus male FNR = {fnr_gap_before:+.4f} ({fnr_gap_before*100:+.2f} percentage points).')
print('A positive FNR gap means more positive-label female examples were missed at threshold 0.5. Compare the chart with the overall rate; similar AUC alone does not imply equal error rates.')
mark_task('task_1_group_metrics', group_metrics=base_model_eval_result.to_dict(orient='records'), female_minus_male_auc=auc_gap_before, female_minus_male_fnr=fnr_gap_before)

In [ ]:
# Instructor numerical values are illustrative. Task 1's measured answer above uses this run.

## The fairness issue
A false negative here is a positive example (annual income **greater than** $50,000) predicted negative at threshold 0.5. A demographic FNR gap relates to [equality of opportunity](https://developers.google.com/machine-learning/glossary#equality-of-opportunity).

Use MinDiff on positive-label Female and Male subsets to penalize differences in their prediction distributions. Measure the resulting gap and overall AUC; improvement is not guaranteed.


### Task 2: Create Positively-Labeled Subsets
Using the training set:


1.   Add a line of code that creates a subset of the training set only containing positive `Female` examples.
2.   Add another line of code that creates a subset of the training set but only containing positive `Male ` examples.



In [ ]:
sensitive_group_pos = acs_train_df.loc[
    (acs_train_df[SENSITIVE_ATTRIBUTE_KEY]==2.0) & (acs_train_df[LABEL_KEY]==1)]
non_sensitive_group_pos = acs_train_df.loc[
    (acs_train_df[SENSITIVE_ATTRIBUTE_KEY]==1.0) & (acs_train_df[LABEL_KEY]==1)]
assert sensitive_group_pos[LABEL_KEY].eq(1).all()
assert non_sensitive_group_pos[LABEL_KEY].eq(1).all()

assert sensitive_group_pos[SENSITIVE_ATTRIBUTE_KEY].eq(2.0).all()
assert non_sensitive_group_pos[SENSITIVE_ATTRIBUTE_KEY].eq(1.0).all()
assert len(sensitive_group_pos) >= 50 and len(non_sensitive_group_pos) >= 50
print('Task 2 positive-label groups:', {'Female':len(sensitive_group_pos), 'Male':len(non_sensitive_group_pos)})
mark_task('task_2_positive_sensitive_groups', female_positive_rows=len(sensitive_group_pos), male_positive_rows=len(non_sensitive_group_pos))

In [ ]:
#@title Double-Click to View a Possible Answer { display-mode: "form" }

# A pandas DataFrame offers many approaches when it comes to indexing and
# selecting rows. One approach is by using boolean indexing
# (e.g., df[df['col'] == value]) as demonstrated in the following code:
sensitive_group_pos = acs_train_df[
    (acs_train_df[SENSITIVE_ATTRIBUTE_KEY] == 2.0) & (acs_train_df[LABEL_KEY] == 1)]
non_sensitive_group_pos = acs_train_df[
    (acs_train_df[SENSITIVE_ATTRIBUTE_KEY] == 1.0) & (acs_train_df[LABEL_KEY] == 1)]

# To learn more, visit: https://pandas.pydata.org/docs/user_guide/indexing.html

print(len(sensitive_group_pos),
      'positively labeled sensitive group examples')
print(len(non_sensitive_group_pos),
      'positively labeled non-sensitive group examples')

## Preparing MinDiff Data
With the sensitive and non-sensitive subsets defined, attempts can now be made at equalizing the distributions between them using [MinDiff](https://www.tensorflow.org/responsible_ai/model_remediation/min_diff/guide/mindiff_overview). MinDiff is one of the [TensorFlow Model Remediation](https://www.tensorflow.org/responsible_ai/model_remediation) techniques used to balance error rates (or, in this exercise, the FNR) between demographic groups (`Male` and `Female`) by penalizing distributional differences during training; hence, MinDiff, or *minimizing the differences*.

As [specified earlier](#scrollTo=6FyuAZ-BYRXU) in this exercise, [`MinDiffModel`](https://www.tensorflow.org/responsible_ai/model_remediation/api_docs/python/model_remediation/min_diff/keras/MinDiffModel) requires [`tf.data.Dataset`](https://www.tensorflow.org/api_docs/python/tf/data/Dataset) as the input. The following code cell will use that same helper function to convert the subsets into `tf.data.Dataset`.

In [ ]:
# Repeat auxiliary group streams while original training data stays finite.
# pack_min_diff_data uses Dataset.zip; without repetition it stops at the shorter
# positive-label group and silently omits most original training batches.
MIN_DIFF_BATCH_SIZE = 50
sensitive_group_ds = dataframe_to_dataset(sensitive_group_pos)
non_sensitive_group_ds = dataframe_to_dataset(non_sensitive_group_pos)
sensitive_group_batches = sensitive_group_ds.batch(MIN_DIFF_BATCH_SIZE, drop_remainder=True).repeat()
non_sensitive_group_batches = non_sensitive_group_ds.batch(MIN_DIFF_BATCH_SIZE, drop_remainder=True).repeat()

### Task 3: Packing the Datasets for MinDiff Model
Now that the subsets are prepared, they must be packed into a single dataset along with the original training set, which will then be passed along to the `MinDiffModel` for training.

To advance, add a line of code that uses [`min_diff.keras.utils.pack_min_diff_data()`](https://www.tensorflow.org/responsible_ai/model_remediation/api_docs/python/model_remediation/min_diff/keras/utils/pack_min_diff_data) to pack the original dataset, the sensitive subset, and the nonsensitive subset.

**Note:** All datasets must be batched before packing them together to prevent errors downstream as a consequence of unintended input tensor shapes.

In [ ]:
acs_train_min_diff_ds = min_diff.keras.utils.pack_min_diff_data(
    original_dataset=acs_train_batches,
    sensitive_group_dataset=sensitive_group_batches,
    nonsensitive_group_dataset=non_sensitive_group_batches)

original_batch_count = int(tf.data.experimental.cardinality(acs_train_batches).numpy())
packed_batch_count = int(tf.data.experimental.cardinality(acs_train_min_diff_ds).numpy())
assert original_batch_count == packed_batch_count == 13316
print('Task 3 packed original training batches per epoch:', packed_batch_count)
mark_task('task_3_pack_complete_training_data', original_batches=original_batch_count, packed_batches=packed_batch_count, original_train_rows=len(acs_train_df))

In [ ]:
# The original Task 3 answer is implemented above. Repeating only auxiliary streams preserves all 13,316 original training batches per epoch.

## Train MinDiff Model
With the data preparation complete, the base model can now be wrapped in the [`MinDiffModel`](https://www.tensorflow.org/responsible_ai/model_remediation/api_docs/python/model_remediation/min_diff/keras/MinDiffModel) and compiled. Configuring the MinDiff model is no different from how the base model was configured [earlier](#scrollTo=tWi8a1O5OnG8).

In [ ]:
# Wrap the original model in a MinDiffModel.
min_diff_model = min_diff.keras.MinDiffModel(
    original_model=base_model,
    loss=min_diff.losses.MMDLoss(),
    loss_weight=1)

# Compile the model after wrapping the original model.
min_diff_model.compile(
    optimizer='adam',
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=False),
    metrics=[tf.keras.metrics.BinaryAccuracy(name='accuracy'),
             tf.keras.metrics.AUC(name='auc')])

As for training, just remember to pass in the packed dataset.

**NOTE:** *The following cell may take approximately 5 minutes to run.*

In [ ]:
# Train the original MinDiff model for all ten epochs on complete packed data.
min_diff_history = min_diff_model.fit(acs_train_min_diff_ds, epochs=EPOCHS, verbose=2)
assert len(min_diff_history.epoch) == 10
pd.DataFrame(min_diff_history.history).to_csv(RESULTS / 'tf_mindiff_remediated_history.csv', index=False)
min_diff_model.original_model.save_weights(str(MODELS / 'tf_mindiff_remediated.weights.h5'))
mark_task('mindiff_training_10_epochs', epochs=len(min_diff_history.epoch), packed_batches_per_epoch=packed_batch_count)

## Evaluate MinDiff Model

When it comes to overall performance, the results of the MinDiff model should be somewhat similar to the base model. Of course, the accuracy or AUC might slightly vary when compared to the base model.

In [ ]:
min_diff_test_metrics = min_diff_model.evaluate(acs_test_batches, return_dict=True, verbose=2)
pd.DataFrame([min_diff_test_metrics]).to_csv(RESULTS / 'tf_mindiff_remediated_test_metrics.csv', index=False)

## Evaluating for Fairness Using Remediated Model
With the MinDiff model now trained, the same steps used to evaluate the base model with Fairness Indicators can be applied to the MinDiff model as well.

Begin by generating the predictions — this time passing the test set as an input argument to the MinDiff model.



In [ ]:
# Generate MinDiff output predictions using the test set.
min_diff_model_predictions = min_diff_model.predict(
    acs_test_batches, batch_size=BATCH_SIZE)

assert min_diff_model_predictions.reshape(-1).shape == (len(acs_test_df),)
mark_task('mindiff_test_predictions', test_rows=len(acs_test_df), test_metrics={k:float(v) for k,v in min_diff_test_metrics.items()})

Same as before, append the predictions as a column onto the DataFrame, then pass it onto Fairness Indicators for evaluation. Note that the configuration used for the base model remains the same for the MinDiff model, which is why it is not being redefined below.

**NOTE:** *The following cell may take 5—10 minutes to run.*

In [ ]:
min_diff_model_analysis = acs_test_df.copy()
min_diff_model_analysis[SENSITIVE_ATTRIBUTE_KEY] = min_diff_model_analysis[SENSITIVE_ATTRIBUTE_KEY].map(SENSITIVE_ATTRIBUTE_VALUES)
min_diff_model_analysis[PREDICTION_KEY] = min_diff_model_predictions.reshape(-1)
min_diff_model_eval_result = fairness_table(min_diff_model_analysis)

### Task 4: Review MinDiff results
Compare baseline and MinDiff AUC/FNR on identical held-out rows.

1. Did overall AUC increase or decrease in this run?
2. Did the female-minus-male FNR gap shrink or grow? How did each group's FNR change?

Use the measured comparison below; instructor numerical examples are not results from this run.


In [ ]:
display(min_diff_model_eval_result)
min_diff_model_eval_result.to_csv('results/tf_mindiff_after_metrics.csv',index=False)

plot_fairness_metrics(min_diff_model_eval_result, 'MinDiff')
comparison = base_model_eval_result.merge(min_diff_model_eval_result, on='group', suffixes=('_baseline','_mindiff'), validate='one_to_one')
assert (comparison['count_baseline'] == comparison['count_mindiff']).all()
comparison['auc_change'] = comparison['auc_mindiff'] - comparison['auc_baseline']
comparison['fnr_change'] = comparison['false_negative_rate_mindiff'] - comparison['false_negative_rate_baseline']
display(comparison[['group','count_baseline','auc_baseline','auc_mindiff','auc_change', 'false_negative_rate_baseline','false_negative_rate_mindiff','fnr_change']])
comparison.to_csv(RESULTS / 'tf_mindiff_comparison.csv', index=False)
fnr_gap_after = group_gap(min_diff_model_eval_result, 'false_negative_rate')
overall_auc_change = float(comparison.set_index('group').loc['All','auc_change'])
absolute_gap_change = abs(fnr_gap_after) - abs(fnr_gap_before)
print(f'Task 4 measured answer: overall AUC change = {overall_auc_change:+.4f}; female-minus-male FNR gap changed from {fnr_gap_before:+.4f} to {fnr_gap_after:+.4f}; absolute disparity change = {absolute_gap_change:+.4f}.')
print('A negative absolute disparity change indicates a smaller FNR gap. Read the per-group table to see whether this happened by improving or worsening each group.')
pd.DataFrame({'row':acs_test_df.index,'observed':acs_test_df[LABEL_KEY].to_numpy(), 'sex':acs_test_df[SENSITIVE_ATTRIBUTE_KEY].to_numpy(), 'baseline_probability':base_model_predictions.reshape(-1), 'mindiff_probability':min_diff_model_predictions.reshape(-1)}).to_csv(RESULTS / 'tf_mindiff_test_predictions.csv', index=False)
mark_task('task_4_measured_comparison', overall_auc_change=overall_auc_change, female_minus_male_fnr_before=fnr_gap_before, female_minus_male_fnr_after=fnr_gap_after, absolute_fnr_gap_change=absolute_gap_change, comparison=comparison.to_dict(orient='records'))

In [ ]:
# Instructor numerical examples are illustrative. Task 4's measured answer above can show improvement or a tradeoff.

## Considerations
This exercise demonstrated an approachable way to train and evalute a model for fairness by expanding on key concepts taught throughout MLCC. The tasks in this notebook provided an example of how a gap in a performance metric between two demographic groups could be a signal that the model may have unfair skews.

However, as discussed in other sections in MLCC, real-world production ML systems are large ecosystems — and the model is just a component of it. That means there are a lot of contributing factors (or confounds) that could affect model performance. Furthermore, there are numerous social and technical processes, known and unknown, that underpin and surround ML and AI technologies. Achieving equality on a particular metric alone does not ensure that the model itself is overall fair.

In practice, there is an assumption that the features used to compare performance across sensitive attributes are readily available, or can be accurately inferred. In actuality, datasets do not often include sensitive attributes, and it is generally not a good idea to impute them.





## Save computational progress
Run the final cell after every exercise cell. It checks the required computational stages ran in this session, saves timestamped evidence, and synchronizes results/models to Drive. Save the notebook itself to retain its cell outputs. Website quiz/exercise progress is tracked separately by Google.


In [ ]:
finish_run(COLAB_CONTEXT)